In [ ]:
# TGraph records carry attributes; convert to geometry only when needed.
def _as_topology(value):
    from topologicpy.Topology import Topology
    from topologicpy.Vertex import Vertex
    if isinstance(value, (list, tuple)):
        return [_as_topology(item) for item in value]
    if not isinstance(value, dict) or "index" not in value or "dictionary" not in value:
        return value
    representation = value.get("representation")
    attributes = value["dictionary"]
    if representation is None and "src" not in value:
        representation = Vertex.ByCoordinates(
            attributes.get("x", 0), attributes.get("y", 0), attributes.get("z", 0))
    if representation is None:
        raise ValueError("This graph record has no geometry representation")
    return Topology.SetDictionary(representation, attributes)


def _mesh_data(graph):
    from topologicpy.TGraph import TGraph
    data = TGraph.MeshData(graph)
    data["vertexDictionaries"] = [v["dictionary"] for v in TGraph.Vertices(graph)]
    data["edgeDictionaries"] = [e["dictionary"] for e in _edge_records(graph)]
    return data


def _edge_records(graph):
    from topologicpy.TGraph import TGraph
    records = TGraph.Edges(graph)
    if any(record.get("representation") is None for record in records):
        geometry = TGraph.Edges(graph, asTopologic=True, segmentCurves=False)
        if len(geometry) != len(records):
            raise ValueError("Could not generate one geometry representation per graph edge")
        for record, edge in zip(records, geometry):
            record["representation"] = edge
    return records


In [ ]:
# pip install openai chromadb numpy topologicpy sentence-transformers accelerate


In [ ]:
# Import TopologicPy modules. This is not needed on other computers
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

In [ ]:
from topologicpy.Edge import Edge
from topologicpy.CellComplex import CellComplex
from topologicpy.TGraph import TGraph
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
print("This jupyter notebook requires topologicpy v.0.8.20 or newer")
print(Helper.Version())

In [ ]:
c = CellComplex.Prism(width=2, length=2, height=2)
cells = Topology.Cells(c)
for i, cell in enumerate(cells):
    d = Dictionary.ByKeysValues(["id", "category"], ["c_"+str(i+1), "cell"])
    cell = Topology.SetDictionary(cell, d)
faces = Topology.Faces(_as_topology(c))
for i, face in enumerate(faces):
    d = Dictionary.ByKeysValues(["id", "category"], ["f_"+str(i+1), "face"])
    face = Topology.SetDictionary(face, d)
supergraph = TGraph.ByTopology(c, toExteriorTopologies=True)
vertices = TGraph.Vertices(supergraph)
edges = _edge_records(supergraph)

subgraph = TGraph.ByVerticesEdges(vertices, edges[:-10])
subgraph = TGraph.RemoveIsolatedVertices(subgraph)


# Modify super graph by adding intermediate vertices
new_edges = []
for i, edge in enumerate(edges):
    sv = Edge.StartVertex(_as_topology(edge))
    c = Topology.Centroid(_as_topology(edge))
    c = Topology.Translate(_as_topology(c), 0, 0, 0.1)
    d = Dictionary.ByKeysValues(["id", "category"], ["n_"+str(i+1), "centroid"])
    c = Topology.SetDictionary(c, d)
    ev = Edge.EndVertex(_as_topology(edge))
    e1 = Edge.ByVertices(_as_topology(sv), _as_topology(c))
    e2 = Edge.ByVertices(_as_topology(c), _as_topology(ev))
    new_edges.append(e1)
    new_edges.append(e2)
    vertices.append(c)
supergraph = TGraph.ByVerticesEdges(vertices, new_edges)
# Add more edges to complete the supergraph
#supergraph = Graph.Complete(supergraph)

print("The supergraph")
Topology.Show(supergraph, vertexSize=12, vertexLabelKey="id", showVertexLabel=True)
print("The subgraph")
Topology.Show(TGraph.Reshape(subgraph), vertexSize=12, vertexLabelKey="id", showVertexLabel=True)

In [ ]:
import chromadb
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from topologicpy.TGraph import TGraph
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Dictionary import Dictionary
from topologicpy.Topology import Topology

# Initialize embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Initialize ChromaDB
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name='building_graph')

# --- Initialize free local Hugging Face model ---
# tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")
# model = AutoModelForCausalLM.from_pretrained(
#     "mistralai/Mistral-7B-Instruct-v0.2",
#     device_map='auto',
#     load_in_4bit=True
# )

# tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
# model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

tokenizer = AutoTokenizer.from_pretrained("tiiuae/falcon-7b-instruct")
model = AutoModelForCausalLM.from_pretrained(
    "tiiuae/falcon-7b-instruct",
    device_map='auto'
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.1,
    clean_up_tokenization_spaces=False
)

# --- Function to convert Topologic graph to texts ---
def graph_to_texts(graph, vertexLabelKey="id", mantissa=6):
    mesh_data = _mesh_data(graph)
    vertices = mesh_data['vertices']
    edges = mesh_data['edges']
    v_dicts = mesh_data['vertexDictionaries']
    e_dicts = mesh_data['edgeDictionaries']
    texts = []
    for i, vertex in enumerate(vertices):
        clean_str = str(v_dicts[i]).translate(str.maketrans('', '', "'\"{}"))
        text = "Vertex: "+clean_str+" Coordinates: "+str(vertex)
        #text = f"Vertex Label: {vertex_label}, Category: {vertex_category}, Coordinates: {coordinates}"
        texts.append(text)

    for i, edge in enumerate(edges):
        edge_dict = e_dicts[i]
        clean_str = str(edge_dict).translate(str.maketrans('', '', "'\"{}"))
        start_dict = v_dicts[edge[0]]
        start_label = start_dict.get(vertexLabelKey, 'Unnamed Vertex')
        end_dict = v_dicts[edge[1]]
        end_label = end_dict.get(vertexLabelKey, 'Unnamed Vertex')
        text = "Edge: "+clean_str+f", connects {start_label} to {end_label}"
        texts.append(text)

    return texts

# --- Function to embed texts into ChromaDB ---
def embed_graph(graph):
    texts = graph_to_texts(graph)
    embeddings = embedder.encode(texts).tolist()
    collection.add(
        embeddings=embeddings,
        documents=texts,
        ids=[f"item_{i}" for i in range(len(texts))]
    )

# --- Retrieval function ---
def retrieve_relevant_nodes(query, top_k=3):
    query_embedding = embedder.encode(query).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    return results['documents'][0]

# --- Local model response generation ---
def generate_response(query, retrieved_docs):
    context = "\n".join(retrieved_docs)
    prompt = f"""<s>[INST] You are an assistant helping users explore graph data.

Graph Information:
{context}

User Question:
{query}

Provide a concise and informative answer based only on the above information. [/INST]"""

    response = generator(prompt)[0]['generated_text']
    answer = response.split('[/INST]')[-1].strip()
    return answer

# --- Main GraphRAG function ---
def ask_graph_rag(query):
    retrieved_docs = retrieve_relevant_nodes(query)
    response = generate_response(query, retrieved_docs)
    return response





In [ ]:
# --- Example Usage ---
if __name__ == '__main__':
    # TODO: Replace this with your actual Topologic graph
    #my_graph = Graph.ByVerticesEdges([], [])  # Example empty graph, replace accordingly

    # Embed your graph (run this once to populate ChromaDB)
    embed_graph(supergraph)

    # User asks a question
    #user_query = "How many edges are there?"
    #answer = ask_graph_rag(user_query)

    #print("Answer:", answer)


In [ ]:
# User asks a question
user_query = "How many edges are connected to c_1?"
answer = ask_graph_rag(user_query)

print("Answer:", answer)

In [ ]:
texts = graph_to_texts(supergraph)
print(texts)


In [ ]:
topologies = Topology.ByOBJPath(r"C:\Users\sarwj\Downloads\room_with_ribbon_window.obj")
Topology.Show(topologies)